# Content Studio — one click, a month of Reels

This runs your faceless content engine **in your browser** using free Google Colab — no installing anything on your computer.

**How to use (3 steps):**
1. Open this notebook in [Google Colab](https://colab.research.google.com/) (File → Open notebook → GitHub tab → paste the repo URL), or click the *Open in Colab* badge in the README.
2. Menu: **Runtime → Run all**.
3. Wait a few minutes. A **`.zip`** with your whole month of content (scripts, captions, hashtags, voiceovers, animated previews, and rendered **MP4 videos**) downloads automatically.

> If the repo is **private**, either make it public in GitHub settings (easiest), or add your token in the setup cell where noted.


In [ ]:
#@title 1) Setup — installs video + voiceover tools (~1 min)
import os, shutil
REPO_URL = "https://github.com/BogdanTheCreator/Automated-Instagram-.git"
# For a PRIVATE repo, use a token instead, e.g.:
# REPO_URL = "https://YOUR_TOKEN@github.com/BogdanTheCreator/Automated-Instagram-.git"
if not os.path.exists("/content/app"):
    !git clone --depth 1 $REPO_URL /content/app
%cd /content/app
!apt-get -qq update >/dev/null && apt-get -qq install -y ffmpeg fonts-dejavu-core >/dev/null
!pip -q install edge-tts
print("\nSetup complete. ffmpeg installed:", bool(shutil.which("ffmpeg")))


In [ ]:
#@title 2) Choose your settings
import os
BRAND = "ai_income"  #@param ["ai_income", "privacy", "money", "longevity"]
DAYS = 30            #@param {type:"slider", min:1, max:30, step:1}
SECONDS = 35         #@param {type:"slider", min:15, max:60, step:5}
POINTS = 4           #@param {type:"slider", min:2, max:6, step:1}
RENDER_VIDEOS = True #@param {type:"boolean"}
START = ""           #@param {type:"string"}
# Optional: paste an OpenAI-compatible API key for AI-written scripts.
# Leave blank to use the built-in templates (still produces a full kit).
OPENAI_API_KEY = ""  #@param {type:"string"}
if OPENAI_API_KEY.strip():
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
print(f"Brand={BRAND}  Days={DAYS}  Seconds={SECONDS}  RenderVideos={RENDER_VIDEOS}")


In [ ]:
#@title 3) Generate everything (then it auto-downloads a .zip)
import os, glob, shutil, time
from social.calendar import write_calendar
from social.render import render_from_package

t0 = time.time()
month = write_calendar(brand_key=BRAND, days=DAYS, start=(START or None),
                       seconds=SECONDS, points=POINTS, with_kits=True, out_root="output")
print("Calendar + content kits written to:", month)

if RENDER_VIDEOS:
    pkgs = sorted(glob.glob(os.path.join(month, "kits", "*", "content_package.json")))
    print(f"\nRendering {len(pkgs)} vertical MP4 videos (the slow part)...")
    for i, pkg in enumerate(pkgs, 1):
        out = os.path.join(os.path.dirname(pkg), "reel.mp4")
        r = render_from_package(pkg, out)
        status = "OK  " if r.ok else "skip"
        print(f"  [{i:>2}/{len(pkgs)}] {status} {os.path.basename(os.path.dirname(pkg))[:48]}")

zip_base = month.replace(os.sep, "_")
shutil.make_archive(zip_base, "zip", month)
print(f"\nDone in {time.time()-t0:.0f}s.  Zip: {zip_base}.zip")
try:
    from google.colab import files
    files.download(f"{zip_base}.zip")
except Exception as exc:
    print("Auto-download unavailable; grab the zip from the Files panel (folder icon, left).", exc)


## What you got, and what to do next

Inside the downloaded zip:
- `calendar.csv` — import into Notion / Google Sheets to plan posting.
- `kits/.../reel.mp4` — a ready vertical video for each day (if you left rendering on).
- `kits/.../post.md` — the caption + hook variants to paste into Instagram.
- `kits/.../hashtags.txt` — a copy-paste hashtag set.
- `kits/.../storyboard.html` — open in a browser to preview the Reel.

**Posting workflow:** put the MP4s on your phone (AirDrop / Google Drive / email), open Instagram → new Reel → pick the video, then paste the caption + hashtags from `post.md`. Use the suggested times in `calendar.csv` and keep the series label so the account feels consistent.

**Tip:** add an `OPENAI_API_KEY` in step 2 to turn the template scripts into finished, original scripts — everything else stays the same.
